In [1]:
import os
from pathlib import Path
import numpy as np
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch
import joblib
from torchvision import models
import pandas as pd

# === CONFIG ===
class Config:
    OUTPUT_DIR = 'lucy_working'
    BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model_new_work_flow.pt")

config = Config()

# === DATASET ===
class PreloadedMelDataset(Dataset):
    def __init__(self, mels, labels, sample_names=None):
        self.mels = mels
        self.labels = labels
        self.sample_names = sample_names if sample_names is not None else [f"sample_{i}" for i in range(len(mels))]

    def __len__(self):
        return len(self.mels)

    def __getitem__(self, idx):
        mel_tensor = torch.tensor(self.mels[idx], dtype=torch.float32).unsqueeze(0)
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel_tensor, label_tensor

# === FILE LOADER ===
def _load_file(file):
    try:
        parts = file.stem.split("-")
        if len(parts) < 3:
            raise ValueError("Invalid filename format")
        year, species = parts[0], parts[1]
        mel_spec = np.load(file)
        if mel_spec.ndim != 2:
            raise ValueError("Bad shape")
        return file.stem, mel_spec, species, year, None
    except Exception as e:
        return file.stem, None, None, None, str(e)

# === PARALLEL MEL LOADER ===
def _load_all_mels_parallel(root_dir=None, allowed_years=("2024", "2025"), max_workers=8):
    if root_dir is None:
        root_dir = os.path.join(config.OUTPUT_DIR, "mel_spectrograms")

    root = Path(root_dir)
    files = [file for class_dir in root.iterdir() if class_dir.is_dir() for file in class_dir.glob("*.npy")]

    loaded_data = {}
    sample_to_label = {}
    bad_files = []

    print(f"🧠 Loading {len(files)} files using {max_workers} workers...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_load_file, file): file for file in files}
        for future in as_completed(futures):
            samplename, mel_spec, species, year, err = future.result()
            if err or year not in allowed_years:
                if err:
                    print(f"❌ {samplename} — {err}")
                    bad_files.append(samplename)
                continue
            loaded_data[samplename] = mel_spec
            sample_to_label[samplename] = species

    print(f"\n✅ Loaded {len(loaded_data)} valid samples after filtering years")
    if bad_files:
        print(f"⚠️ Skipped {len(bad_files)} bad samples")

    return loaded_data, sample_to_label

# === DATALOADER PREPARER ===
def prepare_pretraining_dataloaders(
    data_dict, label_dict, 
    train_names, val_names,
    batch_size=32,
    label_encoder=None
):
    if label_encoder is None:
        label_encoder = LabelEncoder()
        encoded_labels = label_encoder.fit_transform(list(label_dict.values()))
    else:
        encoded_labels = label_encoder.transform(list(label_dict.values()))

    sample_to_encoded = dict(zip(label_dict.keys(), encoded_labels))

    train_mels = [data_dict[name] for name in train_names if name in data_dict]
    train_labels = [sample_to_encoded[name] for name in train_names if name in data_dict]
    val_mels = [data_dict[name] for name in val_names if name in data_dict]
    val_labels = [sample_to_encoded[name] for name in val_names if name in data_dict]

    train_ds = PreloadedMelDataset(train_mels, train_labels, sample_names=train_names)
    val_ds = PreloadedMelDataset(val_mels, val_labels, sample_names=val_names)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size)

    return train_dl, val_dl, len(label_encoder.classes_), label_encoder

# === COMBINED DATASET LOADER ===
def load_combined_dataset(batch_size=32):
    loaded_data, sample_to_label = _load_all_mels_parallel(
        root_dir=os.path.join(config.OUTPUT_DIR, "mel_spectrograms"),
        allowed_years=("2024", "2025")
    )

    train_names_saved = np.load(os.path.join(config.OUTPUT_DIR, "train_names.npy"), allow_pickle=True).tolist()
    val_names_saved = np.load(os.path.join(config.OUTPUT_DIR, "val_names.npy"), allow_pickle=True).tolist()

    stripped_loaded_keys = {name.split("-", 1)[-1]: name for name in loaded_data.keys()}

    available_train_names = [v for k, v in stripped_loaded_keys.items() if k not in val_names_saved]

    print("\n📋 Available Training Samples (after matching):")
    print(f"Total: {len(available_train_names)} samples")
    print(f"First 5 training names: {available_train_names[:5]}")

    print("\n📋 Validation Samples (Saved):")
    print(f"Total: {len(val_names_saved)} samples")
    print(f"First 5 validation names: {val_names_saved[:5]}")

    all_available_names = available_train_names + [stripped_loaded_keys[name] for name in val_names_saved if name in stripped_loaded_keys]
    all_species = [sample_to_label[name] for name in all_available_names]

    label_encoder = LabelEncoder()
    label_encoder.fit(all_species)

    train_dl, val_dl, num_classes, label_encoder = prepare_pretraining_dataloaders(
        data_dict=loaded_data,
        label_dict=sample_to_label,
        train_names=available_train_names,
        val_names=[stripped_loaded_keys[name] for name in val_names_saved if name in stripped_loaded_keys],
        batch_size=batch_size,
        label_encoder=label_encoder
    )

    # === Assert no overlap
    train_sample_names = set(train_dl.dataset.sample_names)
    val_sample_names = set(val_dl.dataset.sample_names)
    assert train_sample_names.isdisjoint(val_sample_names), "❌ Train and validation samples overlap!"

    print(f"\n✅ Final Train Samples: {len(train_dl.dataset)}")
    print(f"✅ Final Validation Samples: {len(val_dl.dataset)}")
    print(f"✅ Total Classes: {num_classes}")

    return train_dl, val_dl, num_classes, label_encoder

# === 2025-ONLY DATASET LOADER ===
def load_2025_dataset(batch_size=32):
    loaded_data, sample_to_label = _load_all_mels_parallel(
        root_dir=os.path.join(config.OUTPUT_DIR, "mel_spectrograms"),
        allowed_years=("2025",)
    )

    train_names_saved = np.load(os.path.join(config.OUTPUT_DIR, "train_names.npy"), allow_pickle=True).tolist()
    val_names_saved = np.load(os.path.join(config.OUTPUT_DIR, "val_names.npy"), allow_pickle=True).tolist()
    label_encoder = joblib.load(os.path.join(config.OUTPUT_DIR, "label_encoder.pkl"))

    stripped_loaded_keys = {name.split("-", 1)[-1]: name for name in loaded_data.keys()}

    train_names_fixed = [stripped_loaded_keys[name] for name in train_names_saved if name in stripped_loaded_keys]
    val_names_fixed = [stripped_loaded_keys[name] for name in val_names_saved if name in stripped_loaded_keys]

    print("\n📋 2025 Training Samples (after matching):")
    print(f"Total: {len(train_names_fixed)} samples")
    print(f"First 5 training names: {train_names_fixed[:5]}")

    print("\n📋 2025 Validation Samples (after matching):")
    print(f"Total: {len(val_names_fixed)} samples")
    print(f"First 5 validation names: {val_names_fixed[:5]}")

    train_dl, val_dl, num_classes, label_encoder = prepare_pretraining_dataloaders(
        data_dict=loaded_data,
        label_dict=sample_to_label,
        train_names=train_names_fixed,
        val_names=val_names_fixed,
        batch_size=batch_size,
        label_encoder=label_encoder
    )

    train_sample_names = set(train_dl.dataset.sample_names)
    val_sample_names = set(val_dl.dataset.sample_names)
    assert train_sample_names.isdisjoint(val_sample_names), "❌ Train and validation samples overlap!"

    print(f"\n✅ Final 2025 Train Samples: {len(train_dl.dataset)}")
    print(f"✅ Final 2025 Validation Samples: {len(val_dl.dataset)}")
    print(f"✅ Total Classes (2025 LabelEncoder): {num_classes}")

    return train_dl, val_dl, num_classes, label_encoder


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import ReduceLROnPlateau


# === EVALUATION ===
def evaluate_roc_auc(model, dataloader, device, num_classes):
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    y_true = label_binarize(all_targets, classes=np.arange(num_classes))
    scored_classes = np.where(y_true.sum(axis=0) > 0)[0]

    if len(scored_classes) == 0:
        return 0.0

    try:
        return roc_auc_score(y_true[:, scored_classes], all_probs[:, scored_classes], average="macro")
    except ValueError:
        return 0.0

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        BCE = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE)
        focal = (1 - pt) ** self.gamma * BCE
        return focal.mean()

# === TRAINING LOOP ===
def train_model(
    train_dl, val_dl, num_classes, model, 
    epochs=10, lr=1e-4, save_name="",
    use_focal_loss=False, gamma=2.0
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

    # === Choose loss function ===
    if use_focal_loss:
        loss_fn = FocalLoss(gamma=gamma)
    else:
        loss_fn = nn.CrossEntropyLoss()

    auc_history = []
    best_auc = 0.0

    # === Create history tracker ===
    history = {
        "epoch": [],
        "train_loss": [],
        "val_auc": [],
        "current_lr": []
    }

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for x, y in train_dl:
            x, y = x.to(device), y.to(device)

            # If using focal loss, convert y to one-hot
            if use_focal_loss:
                y = F.one_hot(y, num_classes=num_classes).float()

            optimizer.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Validation
        val_auc = evaluate_roc_auc(model, val_dl, device, num_classes)
        auc_history.append(val_auc)

        # Step scheduler
        scheduler.step(val_auc)
        current_lr = scheduler.optimizer.param_groups[0]['lr']

        # Update history
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(total_loss)
        history["val_auc"].append(val_auc)
        history["current_lr"].append(current_lr)

        # Save best model
        if val_auc > best_auc:
            best_auc = val_auc
            base = os.path.splitext(config.BEST_MODEL_PATH)[0]
            checkpoint_path = base + "_" + save_name + ".pt"
            torch.save(model.state_dict(), checkpoint_path)

        print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss:.2f} — ROC AUC: {val_auc:.4f} — LR: {current_lr:.2e} {'<-- best' if val_auc == best_auc else ''}")

    print(f"✅ Training complete. Best ROC AUC: {best_auc:.4f}")

    # === Save training history ===
    history_df = pd.DataFrame(history)
    history_path = os.path.join(config.OUTPUT_DIR, f"training_history_{save_name}.csv")
    history_df.to_csv(history_path, index=False)
    print(f"📄 Training history saved to: {history_path}")

    return model, auc_history


In [3]:
train_dl_combined, val_dl_combined, num_classes_combined, label_encoder_combined = load_combined_dataset()

# train_dl_2025, val_dl_2025, num_classes_2025, label_encoder_2025 = load_2025_dataset()

🧠 Loading 53023 files using 8 workers...

✅ Loaded 53023 valid samples after filtering years

📋 Available Training Samples (after matching):
Total: 47047 samples
First 5 training names: ['2025-21211-XC913839', '2025-21211-XC882657', '2025-21211-XC882649', '2025-21211-iNat361222', '2025-21211-XC882654']

📋 Validation Samples (Saved):
Total: 5713 samples
First 5 validation names: ['yeofly1-XC250351', 'whbman1-XC268585', '21211-XC913998', 'bugtan-XC480423', 'strher-XC287867']

✅ Final Train Samples: 47047
✅ Final Validation Samples: 5713
✅ Total Classes: 387


In [4]:
print(f"Train samples: {len(train_dl_combined.dataset)}")
print(f"Validation samples: {len(val_dl_combined.dataset)}")

Train samples: 47047
Validation samples: 5713


In [ ]:
class MelResNet(nn.Module):
    def __init__(self, num_classes, backbone_name="resnet34"):
        super().__init__()
        
        # Choose backbone based on the name
        if backbone_name == "resnet18":
            self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        elif backbone_name == "resnet34":
            self.backbone = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        elif backbone_name == "resnet50":
            self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        elif backbone_name == "resnet101":
            self.backbone = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        elif backbone_name == "resnet152":
            self.backbone = models.resnet152(weights=models.ResNet152_Weights.DEFAULT)
        else:
            raise ValueError(f"Unknown backbone: {backbone_name}")
        
        # Adjust the first conv layer for 1 channel input
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Replace the final fully connected layer
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

    def predict(self, x):
        self.eval()
        with torch.no_grad():
            logits = self(x)
            return torch.argmax(logits, dim=1)

In [6]:
model_res18 = MelResNet(num_classes_combined,backbone_name="resnet18")
_, auc_history_res18 = train_model(train_dl_combined, val_dl_combined, num_classes_combined, model_res18, save_name="resnet18_lucy_new_pipeline",epochs=20)

Epoch 1/20 — Loss: 5159.05 — ROC AUC: 0.9436 — LR: 1.00e-04 <-- best
Epoch 2/20 — Loss: 3052.10 — ROC AUC: 0.9621 — LR: 1.00e-04 <-- best
Epoch 3/20 — Loss: 2216.35 — ROC AUC: 0.9661 — LR: 1.00e-04 <-- best
Epoch 4/20 — Loss: 1600.70 — ROC AUC: 0.9752 — LR: 1.00e-04 <-- best
Epoch 5/20 — Loss: 1085.81 — ROC AUC: 0.9682 — LR: 1.00e-04 
Epoch 6/20 — Loss: 651.77 — ROC AUC: 0.9735 — LR: 5.00e-05 
Epoch 7/20 — Loss: 239.34 — ROC AUC: 0.9749 — LR: 5.00e-05 
Epoch 8/20 — Loss: 120.67 — ROC AUC: 0.9764 — LR: 5.00e-05 <-- best
Epoch 9/20 — Loss: 75.64 — ROC AUC: 0.9758 — LR: 5.00e-05 
Epoch 10/20 — Loss: 54.80 — ROC AUC: 0.9728 — LR: 2.50e-05 
Epoch 11/20 — Loss: 29.01 — ROC AUC: 0.9768 — LR: 2.50e-05 <-- best
Epoch 12/20 — Loss: 18.44 — ROC AUC: 0.9760 — LR: 2.50e-05 
Epoch 13/20 — Loss: 19.75 — ROC AUC: 0.9759 — LR: 1.25e-05 
Epoch 14/20 — Loss: 10.96 — ROC AUC: 0.9761 — LR: 1.25e-05 
Epoch 15/20 — Loss: 9.01 — ROC AUC: 0.9766 — LR: 6.25e-06 
Epoch 16/20 — Loss: 7.13 — ROC AUC: 0.9763 — LR: 

In [4]:
import torch
import torch.nn as nn
from torchvision import models

class MelResNetInference(nn.Module):
    def __init__(self, num_classes, backbone_name="resnet18"):
        super().__init__()
        if backbone_name == "resnet18":
            self.backbone = models.resnet18(weights=None)
        elif backbone_name == "resnet34":
            self.backbone = models.resnet34(weights=None)
        elif backbone_name == "resnet50":
            self.backbone = models.resnet50(weights=None)
        elif backbone_name == "resnet101":
            self.backbone = models.resnet101(weights=None)
        elif backbone_name == "resnet152":
            self.backbone = models.resnet152(weights=None)
        else:
            raise ValueError(f"Unknown backbone: {backbone_name}")

        # Adjust first conv layer
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Replace final fully connected layer
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

    def predict(self, x):
        self.eval()
        with torch.no_grad():
            logits = self(x)
            return torch.argmax(logits, dim=1)

    @classmethod
    def load_from_checkpoint(cls, checkpoint_path, num_classes, backbone_name="resnet18", old_num_classes=None):
        model = cls(num_classes, backbone_name=backbone_name)
        state_dict = torch.load(checkpoint_path, map_location="cpu")

        if old_num_classes is not None:
            # Remove old fc weights
            keys_to_remove = [k for k in state_dict.keys() if "fc.weight" in k or "fc.bias" in k]
            for k in keys_to_remove:
                print(f"🔵 Skipping incompatible key during load: {k}")
                del state_dict[k]
            model.load_state_dict(state_dict, strict=False)
            print(f"✅ Loaded backbone and reset fc layer for {num_classes} classes")
        else:
            model.load_state_dict(state_dict)
            print(f"✅ Fully loaded model from {checkpoint_path}")

        model.eval()
        return model
def freeze_backbone(model):
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.backbone.fc.parameters():
        param.requires_grad = True
    print("🧊 Backbone frozen, fc layer trainable.")

def unfreeze_backbone(model):
    for param in model.backbone.parameters():
        param.requires_grad = True
    print("🔥 Backbone unfrozen, full model trainable.")


In [5]:
train_dl_2025, val_dl_2025, num_classes_2025, label_encoder_2025 = load_2025_dataset()

🧠 Loading 53023 files using 8 workers...

✅ Loaded 28564 valid samples after filtering years

📋 2025 Training Samples (after matching):
Total: 22851 samples
First 5 training names: ['2025-creoro1-XC122330', '2025-whtdov-XC497529', '2025-brtpar1-XC934051', '2025-compau-XC900266', '2025-banana-XC215533']

📋 2025 Validation Samples (after matching):
Total: 5713 samples
First 5 validation names: ['2025-yeofly1-XC250351', '2025-whbman1-XC268585', '2025-21211-XC913998', '2025-bugtan-XC480423', '2025-strher-XC287867']

✅ Final 2025 Train Samples: 22851
✅ Final 2025 Validation Samples: 5713
✅ Total Classes (2025 LabelEncoder): 206


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load model backbone (pretrained with more classes)
model = MelResNetInference.load_from_checkpoint(
    checkpoint_path="lucy_working/best_model_new_work_flow_resnet18_lucy_new_pipeline.pt",
    num_classes=num_classes_2025,            # 2025 classes
    backbone_name="resnet18",
    old_num_classes=387         # How many classes the checkpoint originally had hard coded as I have limited memery and clear it for new data.
)
model = model.to(device)

# 2. Freeze backbone initially
freeze_backbone(model)

# 3. Train only fc head for a few epochs
_, auc_history = train_model(
    train_dl=train_dl_2025,
    val_dl=val_dl_2025,
    num_classes=num_classes_2025,
    model=model,
    epochs=10,
    lr=1e-4,
    save_name="resnet18_finetune_head_only"
)



🔵 Skipping incompatible key during load: backbone.fc.weight
🔵 Skipping incompatible key during load: backbone.fc.bias
✅ Loaded backbone and reset fc layer for 206 classes
🧊 Backbone frozen, fc layer trainable.
Epoch 1/10 — Loss: 1809.07 — ROC AUC: 0.9353 — LR: 1.00e-04 <-- best
Epoch 2/10 — Loss: 405.50 — ROC AUC: 0.9681 — LR: 1.00e-04 <-- best
Epoch 3/10 — Loss: 164.07 — ROC AUC: 0.9744 — LR: 1.00e-04 <-- best
Epoch 4/10 — Loss: 89.55 — ROC AUC: 0.9758 — LR: 1.00e-04 <-- best
Epoch 5/10 — Loss: 55.30 — ROC AUC: 0.9761 — LR: 1.00e-04 <-- best
Epoch 6/10 — Loss: 38.46 — ROC AUC: 0.9768 — LR: 1.00e-04 <-- best
Epoch 7/10 — Loss: 26.75 — ROC AUC: 0.9771 — LR: 1.00e-04 <-- best
Epoch 8/10 — Loss: 19.23 — ROC AUC: 0.9771 — LR: 1.00e-04 <-- best
Epoch 9/10 — Loss: 15.27 — ROC AUC: 0.9766 — LR: 5.00e-05 
Epoch 10/10 — Loss: 13.52 — ROC AUC: 0.9769 — LR: 5.00e-05 
✅ Training complete. Best ROC AUC: 0.9771
📄 Training history saved to: lucy_working/training_history_resnet18_finetune_head_only.cs

In [8]:
# 4. Unfreeze full model
unfreeze_backbone(model)

# 5. Train fully fine-tuned model
_, auc_history_full = train_model(
    train_dl=train_dl_2025,
    val_dl=val_dl_2025,
    num_classes=num_classes_2025,
    model=model,
    epochs=15,
    lr=1e-5,
    save_name="resnet18_finetune_full"
)

🔥 Backbone unfrozen, full model trainable.
Epoch 1/15 — Loss: 10.19 — ROC AUC: 0.9759 — LR: 1.00e-05 <-- best
Epoch 2/15 — Loss: 7.48 — ROC AUC: 0.9760 — LR: 1.00e-05 <-- best
Epoch 3/15 — Loss: 5.68 — ROC AUC: 0.9752 — LR: 1.00e-05 
Epoch 4/15 — Loss: 5.09 — ROC AUC: 0.9753 — LR: 5.00e-06 
Epoch 5/15 — Loss: 3.88 — ROC AUC: 0.9752 — LR: 5.00e-06 
Epoch 6/15 — Loss: 3.60 — ROC AUC: 0.9755 — LR: 2.50e-06 
Epoch 7/15 — Loss: 4.09 — ROC AUC: 0.9758 — LR: 2.50e-06 
Epoch 8/15 — Loss: 3.18 — ROC AUC: 0.9753 — LR: 1.25e-06 
Epoch 9/15 — Loss: 2.86 — ROC AUC: 0.9751 — LR: 1.25e-06 
Epoch 10/15 — Loss: 3.16 — ROC AUC: 0.9753 — LR: 6.25e-07 
Epoch 11/15 — Loss: 2.48 — ROC AUC: 0.9755 — LR: 6.25e-07 
Epoch 12/15 — Loss: 3.30 — ROC AUC: 0.9755 — LR: 3.13e-07 
Epoch 13/15 — Loss: 3.15 — ROC AUC: 0.9757 — LR: 3.13e-07 
Epoch 14/15 — Loss: 3.09 — ROC AUC: 0.9760 — LR: 1.56e-07 
Epoch 15/15 — Loss: 2.61 — ROC AUC: 0.9756 — LR: 1.56e-07 
✅ Training complete. Best ROC AUC: 0.9760
📄 Training history sav